# Selector v3: E5 vs EmbeddingGemma 2
**Ariel Lax** | Corpus2GeoRAG / CollectCorpus

The v3 selector (40 pages, 3,009 cleaned paragraphs) with two embedding models. Everything else is identical:
the training paragraphs, the classifier (logistic regression, C=10), the thresholds and the exam pages.

| | [multilingual-e5-large](https://huggingface.co/intfloat/multilingual-e5-large) (current) | [embeddinggemma-2](https://huggingface.co/google/embeddinggemma-2) |
|---|---|---|
| size (text) | 560M | 270M |
| vector | 1024 | 768 (can be cut to 512/256/128) |
| max length | 512 tokens | 8,192 tokens (here capped at 2,048) |
| prompt | `query: ` | `task: classification \| query: ` |

Measured: (1) cross-validation by page on the training paragraphs, (2) AUC on the exam pages, (3) the three zones,
(4) speed. Gemma is also tested with its vectors cut to 256 (smaller and faster to store).

### Before running
1. Nothing new to upload: `paragraphs_v3.csv`, `eval_pages.jsonl` and the E5 vectors are already in `MyDrive/selector`.
2. Runtime → Change runtime type → **T4 GPU**. Start from a **fresh runtime** (Runtime → Disconnect and delete runtime),
   then Runtime → **Run all**, and approve the Drive connection. About 30–45 minutes; if it disconnects, Run all again
   and it continues from the saved chunks.
3. At the end, download from `MyDrive/selector`: **`compare_summary.csv`**, **`compare_auc.csv`**,
   **`compare_exam_scores.csv`**.

## Setup

In [ ]:
# EmbeddingGemma 2 needs recent versions (run on a fresh runtime, before anything imports transformers)
%pip install -q -U sentence-transformers transformers

In [ ]:
import hashlib, json, os, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import sentence_transformers, transformers
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none (this will be slow)',
      '| sentence-transformers', sentence_transformers.__version__, '| transformers', transformers.__version__)

In [ ]:
LOCAL = False   # True = small local test
GEMMA_MODEL = 'google/embeddinggemma-2'

if LOCAL:
    WORK = 'data/selector/compare_test'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/selector'
for f in ['paragraphs_v3.csv', 'eval_pages.jsonl']:
    assert os.path.exists(f'{WORK}/{f}'), f'Missing {f} in {WORK}'
print('working folder:', WORK)

## The selector code (`build_training_set.py`, `selector.py`)

In [ ]:
MIN_CHARS = 50
SKIP_SECTIONS = {"References", "See also", "External links", "Further reading", "Notes", "Bibliography",
                 "Sources", "Citations", "Footnotes", "Explanatory notes", "Works cited"}

In [ ]:
def split_paragraphs(content):
    """[(section path, paragraph), ...] following the rules in the module docstring."""
    out, path, skip = [], [], False
    for line in content.split("\n"):
        line = line.strip()
        heading = re.match(r"^(=+)\s*(.*?)\s*=+$", line)
        if heading:
            level = len(heading.group(1)) - 1          # "== X ==" is level 1
            path = path[:level - 1] + [heading.group(2)]
            if level == 1:
                skip = heading.group(2) in SKIP_SECTIONS
            continue
        if not skip and len(line) >= MIN_CHARS:
            out.append((" > ".join(path) or "(intro)", line))
    return out

In [ ]:
E5_MODEL = "intfloat/multilingual-e5-large"
PARAGRAPH_THRESHOLD = 0.7
ACCEPT_THRESHOLD = 0.30
REJECT_THRESHOLD = 0.10
MIN_RELATED_PARAGRAPHS = 7

In [ ]:
class Selector:
    def __init__(self, encoder, model_name=E5_MODEL, paragraph_threshold=PARAGRAPH_THRESHOLD,
                 accept_threshold=ACCEPT_THRESHOLD, reject_threshold=REJECT_THRESHOLD,
                 min_related_paragraphs=MIN_RELATED_PARAGRAPHS, clf=None):
        self.encoder = encoder              # a sentence_transformers.SentenceTransformer (E5)
        self.model_name = model_name
        self.paragraph_threshold = paragraph_threshold
        self.accept_threshold = accept_threshold
        self.reject_threshold = reject_threshold
        self.min_related_paragraphs = min_related_paragraphs
        self.clf = clf

    # ---------------------------------------------------------------- vectors
    def embed(self, texts, batch_size=64, show_progress_bar=False):
        """E5 vectors; E5 expects the "query: " prefix for this kind of task."""
        return self.encoder.encode(["query: " + t for t in texts], batch_size=batch_size,
                                   normalize_embeddings=True, show_progress_bar=show_progress_bar)

    # ---------------------------------------------------------------- training
    def fit(self, paragraphs, labels, vectors=None):
        """Logistic regression on the paragraph vectors. class_weight="balanced" evens out the
        different number of paragraphs in the two groups."""
        x = self.embed(paragraphs, show_progress_bar=True) if vectors is None else vectors
        self.clf = LogisticRegression(C=10, max_iter=2000, class_weight="balanced").fit(x, labels)
        return self

    # ---------------------------------------------------------------- scoring
    def paragraph_scores(self, paragraphs, vectors=None):
        """Probability (0..1) that each paragraph is related."""
        if vectors is None:
            if not paragraphs:
                return np.array([])
            vectors = self.embed(paragraphs)
        return self.clf.predict_proba(vectors)[:, 1]

    def page_score(self, paragraph_probs, paragraph_threshold=None):
        """Share of the page's paragraphs that count as related (0..1)."""
        t = self.paragraph_threshold if paragraph_threshold is None else paragraph_threshold
        return float(np.mean(np.asarray(paragraph_probs) >= t)) if len(paragraph_probs) else 0.0

    def decision(self, score, related=0):
        """"accept", "review" or "reject" for a page score (and its number of related paragraphs)."""
        if score >= self.accept_threshold or (self.min_related_paragraphs and related >= self.min_related_paragraphs):
            return "accept"
        return "reject" if score < self.reject_threshold else "review"

    def score_page(self, text):
        """Everything for one page's raw text: {"score", "decision", "paragraphs", "related_paragraphs"}."""
        paragraphs = [p for _, p in split_paragraphs(text)]
        probs = self.paragraph_scores(paragraphs)
        score = self.page_score(probs)
        related = int((probs >= self.paragraph_threshold).sum())
        return {"score": round(score, 3), "decision": self.decision(score, related), "paragraphs": len(paragraphs),
                "related_paragraphs": related}

    # ---------------------------------------------------------------- save / load
    def save(self, folder):
        """The classifier and the settings; the E5 model itself is downloaded by name when loading."""
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        joblib.dump(self.clf, folder / "selector_logreg.joblib")
        (folder / "selector_config.json").write_text(json.dumps({
            "model_name": self.model_name, "paragraph_threshold": self.paragraph_threshold,
            "accept_threshold": self.accept_threshold, "reject_threshold": self.reject_threshold,
            "min_related_paragraphs": self.min_related_paragraphs}, indent=2),
            encoding="utf-8")

    @classmethod
    def load(cls, folder, encoder=None):
        folder = Path(folder)
        config = json.loads((folder / "selector_config.json").read_text(encoding="utf-8"))
        if encoder is None:
            from sentence_transformers import SentenceTransformer
            encoder = SentenceTransformer(config["model_name"])
        return cls(encoder, clf=joblib.load(folder / "selector_logreg.joblib"), **config)

The same selector with Gemma: only the vectors change (Gemma's own classification prompt, as its model card says).

In [ ]:
class GemmaSelector(Selector):
    def embed(self, texts, batch_size=32, show_progress_bar=False):
        return self.encoder.encode(texts, prompt_name='Classification', batch_size=batch_size,
                                   normalize_embeddings=True, show_progress_bar=show_progress_bar)

def cut(x, dims):
    """Gemma vectors cut to their first `dims` numbers (Matryoshka) and normalized again."""
    x = x[:, :dims]
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def roc_auc(y, s):
    ranks = pd.Series(s).rank().to_numpy(); pos = y == 1
    if pos.all() or (~pos).all():
        return float('nan')
    return float((ranks[pos].sum() - pos.sum() * (pos.sum() + 1) / 2) / (pos.sum() * (~pos).sum()))

def page_counts(probs, para_page, n_pages, t):
    """(related paragraphs, share) per page at paragraph threshold t."""
    related = np.bincount(para_page, weights=probs >= t, minlength=n_pages).astype(int)
    total = np.bincount(para_page, minlength=n_pages)
    return related, np.divide(related, total, out=np.zeros(n_pages), where=total > 0)

def zones(share, related):
    return np.where((share >= ACCEPT_THRESHOLD) | (related >= MIN_RELATED_PARAGRAPHS), 'accept',
                    np.where(share < REJECT_THRESHOLD, 'reject', 'review'))

def cached_chunks(texts, folder, embed, chunk=5000):
    """Vectors for all texts, computed in chunks saved to `folder`; chunks already there are reused."""
    folder = Path(folder); folder.mkdir(exist_ok=True)
    parts = []
    for start in range(0, len(texts), chunk):
        path = folder / f'{start:06d}.npy'
        if not path.exists():
            np.save(path, embed(texts[start:start + chunk]))
            print('  computed vectors', min(start + chunk, len(texts)), '/', len(texts), flush=True)
        parts.append(np.load(path))
    return np.vstack(parts)

## 1. The two models

In [ ]:
e5 = SentenceTransformer(E5_MODEL, device=DEVICE)
gemma = SentenceTransformer(GEMMA_MODEL, device=DEVICE,
                            config_kwargs={'vision_config': None, 'audio_config': None},   # text only
                            model_kwargs={'torch_dtype': torch.float32})                   # not fp16 (model card)
gemma.max_seq_length = 2048
selectors = {'e5': Selector(e5), 'gemma': GemmaSelector(gemma, model_name=GEMMA_MODEL)}
print('Gemma prompt:', repr(gemma.prompts['Classification']))

## 2. Training vectors (the same 3,009 paragraphs)

In [ ]:
train = pd.read_csv(f'{WORK}/paragraphs_v3.csv', encoding='utf-8-sig')
if LOCAL:
    train = train.groupby('page').head(8)
fingerprint = hashlib.md5('\n'.join(train.text).encode('utf-8')).hexdigest()[:8]
train_x = {}
for name, cache in [('e5', f'{WORK}/train_vectors_e5_v3_{fingerprint}.npy'),           # from SelectorV3.ipynb
                    ('gemma', f'{WORK}/train_vectors_gemma2_v3_{fingerprint}.npy')]:
    if not os.path.exists(cache):
        np.save(cache, selectors[name].embed(train.text.tolist(), show_progress_bar=True))
    train_x[name] = np.load(cache)
train_x['gemma-256'] = cut(train_x['gemma'], 256)
y, groups = train.label.to_numpy(), train.page.to_numpy()
print('training paragraphs:', train.label.value_counts().to_dict(), '|', {k: v.shape[1] for k, v in train_x.items()})

## 3. Cross-validation by page (on the training paragraphs)
The 40 pages are split into 5 groups; the classifier trains on 4 and scores the paragraphs of the 5th, which it never saw.
These labels are clean (checked by hand), so this is the most exact measure of how well each model separates
military from non-military paragraphs.

In [ ]:
cv_rows = []
folds = GroupKFold(n_splits=5)
for name, x in train_x.items():
    oof = np.zeros(len(y))
    for tr, te in folds.split(x, y, groups):
        clf = LogisticRegression(C=10, max_iter=2000, class_weight='balanced').fit(x[tr], y[tr])
        oof[te] = clf.predict_proba(x[te])[:, 1]
    pred = oof >= PARAGRAPH_THRESHOLD
    cv_rows.append({'model': name, 'cv_auc': round(roc_auc(y, oof), 3),
                    'cv_precision@0.7': round((y[pred] == 1).mean(), 3),
                    'cv_recall@0.7': round(pred[y == 1].mean(), 3)})
cv = pd.DataFrame(cv_rows).set_index('model')
cv

## 4. The exam: the corpus pages, without the 40 training pages
Same pages and paragraphs as in SelectorV3.ipynb. The E5 vectors are reused; the Gemma vectors are computed in chunks
(`eval_vectors_embeddinggemma-2/`), so a disconnect doesn't lose work.

In [ ]:
pages = [json.loads(l) for l in open(f'{WORK}/eval_pages.jsonl', encoding='utf-8')]
if LOCAL:
    pages = [p for p in pages if p['label'] == 1][:5] + [p for p in pages if p['label'] == 0][:5]
para_page, para_text = [], []
for i, p in enumerate(pages):
    for _, text in split_paragraphs(p['content']):
        para_page.append(i); para_text.append(text)
para_page = np.array(para_page, dtype=int)

eval_x = {'e5': cached_chunks(para_text, Path(WORK) / 'eval_vectors_multilingual-e5-large', selectors['e5'].embed),
          'gemma': cached_chunks(para_text, Path(WORK) / 'eval_vectors_embeddinggemma-2', selectors['gemma'].embed)}
eval_x['gemma-256'] = cut(eval_x['gemma'], 256)
assert all(len(x) == len(para_text) for x in eval_x.values()), 'cached exam vectors do not match the exam pages'

keep = np.array([p['title'] not in set(train.page) for p in pages])
labels = np.array([p['label'] for p in pages])
print(f'{len(pages)} exam pages, {keep.sum()} after removing training pages ({len(para_text)} paragraphs)')

In [ ]:
exam = pd.DataFrame({'title': [p['title'] for p in pages], 'label': labels,
                     'label_status': [p['label_status'] for p in pages],
                     'paragraphs': np.bincount(para_page, minlength=len(pages))})[keep]
rows = []
for name, x in eval_x.items():
    clf = LogisticRegression(C=10, max_iter=2000, class_weight='balanced').fit(train_x[name], y)
    pr = clf.predict_proba(x)[:, 1]
    for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
        related, share = page_counts(pr, para_page, len(pages), t)
        rows.append({'model': name, 'paragraph_threshold': t, 'auc': round(roc_auc(labels[keep], share[keep]), 3)})
        if t == PARAGRAPH_THRESHOLD:
            exam[f'{name}_score'], exam[f'{name}_related'] = np.round(share[keep], 3), related[keep]
            exam[f'{name}_zone'] = zones(share[keep], related[keep])
auc = pd.DataFrame(rows).pivot(index='paragraph_threshold', columns='model', values='auc')
auc

The three zones (paragraph ≥ 0.7; accept ≥ 0.30 or ≥ 7 related paragraphs; reject < 0.10):

In [ ]:
models = list(eval_x)
table = pd.concat({n: pd.crosstab(exam[f'{n}_zone'], exam.label.map({1: 'kept (1)', 0: 'removed (0)'}))
                   for n in models}, axis=1).reindex(['accept', 'review', 'reject'])
table

## 5. Speed (the same 1,000 exam paragraphs, after a warm-up)

In [ ]:
sample = para_text[:1000]
speed = {}
for name in ['e5', 'gemma']:
    selectors[name].embed(sample[:64])                      # warm-up
    if DEVICE == 'cuda': torch.cuda.synchronize()
    start = time.time()
    selectors[name].embed(sample)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    speed[name] = round((time.time() - start) * 1000 / len(sample), 1)
speed['gemma-256'] = speed['gemma']
print('seconds per 1,000 paragraphs:', speed)

## 6. Summary

In [ ]:
summary = cv.copy()
summary['exam_auc@0.7'] = auc.loc[PARAGRAPH_THRESHOLD]
summary['best_exam_auc'] = auc.max()
for n in models:
    acc = exam[exam[f'{n}_zone'] == 'accept']
    summary.loc[n, 'accept_pages'] = len(acc)
    summary.loc[n, 'accept_precision'] = round(acc.label.mean(), 3)
    summary.loc[n, 'kept_rejected'] = int(((exam[f'{n}_zone'] == 'reject') & (exam.label == 1)).sum())
    summary.loc[n, 'removed_accepted'] = int(((exam[f'{n}_zone'] == 'accept') & (exam.label == 0)).sum())
summary['dims'] = [train_x[n].shape[1] for n in summary.index]
summary['sec_per_1000'] = [speed[n] for n in summary.index]
summary.to_csv(f'{WORK}/compare_summary.csv')
auc.to_csv(f'{WORK}/compare_auc.csv')
exam.to_csv(f'{WORK}/compare_exam_scores.csv', index=False)
summary

Pages where the models disagree (accept by one, reject by the other), to look at:

In [ ]:
diff = exam[((exam.e5_zone == 'accept') & (exam.gemma_zone == 'reject')) | ((exam.e5_zone == 'reject') & (exam.gemma_zone == 'accept'))]
print(len(diff), 'pages')
diff[['title', 'label', 'paragraphs', 'e5_score', 'e5_zone', 'gemma_score', 'gemma_zone']].head(40)

Download `compare_summary.csv`, `compare_auc.csv` and `compare_exam_scores.csv` from `MyDrive/selector`.